# 📊 Customer Churn Prediction — Exploratory Data Analysis

**Dataset:** IBM Telco Customer Churn  
**Objective:** Understand the data, identify patterns, detect data quality issues, and extract insights to guide preprocessing and model building.

---

## 📋 Analysis Sections
1. Dataset Overview
2. Target Variable Analysis
3. Numerical Columns Analysis
4. Categorical Columns Analysis
5. Missing Values & Data Quality
6. Duplicate Records
7. Churn Patterns & Correlations
8. Key Insights Summary

---
## 🔧 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid', palette='Set2')

# Color palette for churn/no-churn
CHURN_COLORS = {'No': '#2ecc71', 'Yes': '#e74c3c'}

print('✅ Libraries imported successfully')

---
## 📂 2. Load Dataset

In [ ]:
# Load the raw dataset
df = pd.read_csv('../data/raw/dataset.csv')

print(f'✅ Dataset loaded successfully')
print(f'   Shape : {df.shape[0]} rows × {df.shape[1]} columns')
print(f'   Memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Last 5 rows
df.tail()

In [ ]:
# Column names and data types
print('📋 Column Info:')
print('=' * 50)
df.info()

---
## 🎯 3. Target Variable Analysis — `Churn`

**Definition:** Whether the customer left the company within the last month.
- `Yes` = Customer churned (left)
- `No`  = Customer stayed

This is our **binary classification target**. We need to understand its distribution (class balance/imbalance).

In [ ]:
# Value counts
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

print('📊 Target Variable Distribution:')
print('=' * 35)
for label in ['No', 'Yes']:
    print(f'  Churn = {label}: {churn_counts[label]:>5,}  ({churn_pct[label]:.1f}%)')
print(f'  Total       : {len(df):>5,}  (100.0%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Bar chart ---
ax1 = axes[0]
bars = ax1.bar(churn_counts.index, churn_counts.values,
               color=[CHURN_COLORS['No'], CHURN_COLORS['Yes']],
               edgecolor='black', linewidth=0.8, width=0.5)
for bar, count, pct in zip(bars, churn_counts.values, churn_pct.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax1.set_title('Churn Distribution (Count)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Churn', fontsize=12)
ax1.set_ylabel('Number of Customers', fontsize=12)
ax1.set_ylim(0, churn_counts.max() * 1.2)

# --- Pie chart ---
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    churn_counts.values,
    labels=['No Churn', 'Churned'],
    autopct='%1.1f%%',
    colors=[CHURN_COLORS['No'], CHURN_COLORS['Yes']],
    startangle=140,
    explode=(0, 0.07),
    shadow=True
)
for t in autotexts:
    t.set_fontsize(13)
    t.set_fontweight('bold')
ax2.set_title('Churn Proportion', fontsize=14, fontweight='bold')

plt.suptitle('Target Variable: Churn', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../logs/churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 INSIGHT: Class Imbalance detected!')
print(f'   Churned customers are only ~26% of the dataset.')
print(f'   We must use stratified train/test split and F1-score (not Accuracy) as the main metric.')

---
## 🔢 4. Numerical Columns Analysis

The dataset has **3 numerical features**:
| Column | Description |
|--------|------------|
| `tenure` | Number of months the customer has been with the company |
| `MonthlyCharges` | Current monthly charge amount |
| `TotalCharges` | Total amount charged to date (stored as string — data quality issue!) |

In [ ]:
# Fix TotalCharges — it was read as object (string) due to blank spaces
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

print('📊 Descriptive Statistics — Numerical Columns:')
print('=' * 60)
df[numerical_cols].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Numerical Features — Distribution Analysis', fontsize=16, fontweight='bold')

for i, col in enumerate(numerical_cols):
    # Row 1: Histogram with KDE
    ax_hist = axes[0, i]
    for churn_val, color in CHURN_COLORS.items():
        subset = df[df['Churn'] == churn_val][col].dropna()
        ax_hist.hist(subset, bins=30, alpha=0.6, color=color,
                     label=f'Churn={churn_val}', density=True)
    ax_hist.set_title(f'{col} — Histogram by Churn', fontweight='bold')
    ax_hist.set_xlabel(col)
    ax_hist.set_ylabel('Density')
    ax_hist.legend()

    # Row 2: Boxplot by Churn
    ax_box = axes[1, i]
    temp = df[['Churn', col]].dropna()
    groups = [temp[temp['Churn'] == val][col].values for val in ['No', 'Yes']]
    bp = ax_box.boxplot(groups, labels=['No Churn', 'Churned'], patch_artist=True,
                        notch=True, widths=0.4)
    for patch, color in zip(bp['boxes'], [CHURN_COLORS['No'], CHURN_COLORS['Yes']]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax_box.set_title(f'{col} — Boxplot by Churn', fontweight='bold')
    ax_box.set_ylabel(col)

plt.tight_layout()
plt.savefig('../logs/numerical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Mean values by churn
print('📊 Mean values by Churn status:')
print('=' * 50)
print(df.groupby('Churn')[numerical_cols].mean().round(2).to_string())

print('\n💡 INSIGHTS:')
print('  tenure      : Churned customers have LOWER tenure (~18 months vs ~38 months)')
print('  MonthlyCharge: Churned customers pay MORE per month (~$74 vs $61)')
print('  TotalCharges : Churned customers have LOWER total charges (less time with company)')

---
## 🏷️ 5. Categorical Columns Analysis

The dataset has **17 categorical features**. These will need encoding before model training.

In [ ]:
# Identify all categorical columns (excluding customerID and Churn)
categorical_cols = df.select_dtypes(include='object').columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in ['customerID', 'Churn']]

print(f'📋 Categorical Columns ({len(categorical_cols)} total):')
print('=' * 55)
for col in categorical_cols:
    unique_vals = df[col].unique()
    print(f'  {col:<25} → {list(unique_vals)}')

In [ ]:
# Churn rate by each categorical feature
def plot_churn_rate_by_category(df, col, ax):
    """Plot churn rate (%) per category value."""
    churn_rate = df.groupby(col)['Churn'].apply(
        lambda x: (x == 'Yes').sum() / len(x) * 100
    ).reset_index()
    churn_rate.columns = [col, 'Churn Rate (%)']
    churn_rate = churn_rate.sort_values('Churn Rate (%)', ascending=False)

    bars = ax.bar(churn_rate[col].astype(str), churn_rate['Churn Rate (%)'],
                  color=sns.color_palette('Set2', len(churn_rate)),
                  edgecolor='black', linewidth=0.5)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height + 0.5,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(f'Churn Rate by {col}', fontweight='bold', fontsize=11)
    ax.set_ylabel('Churn Rate (%)')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
    ax.set_ylim(0, churn_rate['Churn Rate (%)'].max() * 1.25)
    ax.axhline(y=df['Churn'].eq('Yes').mean()*100, color='red',
               linestyle='--', linewidth=1.2, label='Overall Avg')
    ax.legend(fontsize=8)

# Key categorical features to plot
key_cats = ['Contract', 'InternetService', 'PaymentMethod',
            'TechSupport', 'OnlineSecurity', 'gender',
            'SeniorCitizen', 'Partner', 'Dependents']

fig, axes = plt.subplots(3, 3, figsize=(20, 16))
fig.suptitle('Churn Rate by Categorical Features', fontsize=18, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(key_cats):
    plot_churn_rate_by_category(df, col, axes[i])

plt.tight_layout()
plt.savefig('../logs/categorical_churn_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('💡 KEY INSIGHTS from Categorical Analysis:')
print('=' * 60)

insights = [
    ('Contract',         'Month-to-month contracts churn ~43% vs 2-year at ~3%  → STRONGEST predictor'),
    ('OnlineSecurity',   'No online security → churn ~42% vs 15% with security   → strong predictor'),
    ('TechSupport',      'No tech support → churn ~42% vs 15% with support       → strong predictor'),
    ('InternetService',  'Fiber optic users churn more (~42%) than DSL users (~19%)'),
    ('PaymentMethod',    'Electronic check payers churn most (~45%)'),
    ('SeniorCitizen',    'Seniors (1) churn ~42% vs non-seniors ~24%'),
    ('Partner',          'Customers without partners churn more (~33% vs ~20%)'),
    ('Dependents',       'Customers without dependents churn more (~31% vs ~15%)'),
    ('gender',           'Gender has minimal impact on churn (~same rate for M/F)'),
]

for col, insight in insights:
    print(f'  {col:<20}: {insight}')

---
## ⚠️ 6. Missing Values & Data Quality Issues

In [ ]:
# Missing values after TotalCharges conversion
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
missing_df = missing_df[missing_df['Missing Count'] > 0]

print('🔍 Missing Values Report:')
print('=' * 45)
if missing_df.empty:
    print('  ✅ No missing values found in raw data.')
    print('  ⚠️  BUT: TotalCharges has blank strings ("") for new customers (tenure=0)')
    print('  These become NaN after pd.to_numeric() conversion — need to be handled.')
else:
    print(missing_df.to_string())

In [ ]:
# Visualize missing values
fig, ax = plt.subplots(figsize=(12, 5))
missing_all = (df.isnull().sum() / len(df)) * 100
missing_all = missing_all.sort_values(ascending=False)
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in missing_all.values]
ax.bar(missing_all.index, missing_all.values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_title('Missing Values (%) per Column', fontsize=14, fontweight='bold')
ax.set_ylabel('Missing %')
ax.set_xlabel('Column')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../logs/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Data Quality Issues — full audit
print('🔍 Data Quality Audit:')
print('=' * 60)

# Issue 1: TotalCharges stored as string
raw_df = pd.read_csv('../data/raw/dataset.csv')
print(f'Issue 1 — TotalCharges dtype: {raw_df["TotalCharges"].dtype}')
print(f'          Should be float64, is object (string)')
blanks = (raw_df['TotalCharges'].str.strip() == '').sum()
print(f'          Blank string entries: {blanks} rows (new customers with tenure=0)')
print(f'          Fix: pd.to_numeric(errors="coerce"), then drop NaN rows')
print()

# Issue 2: SeniorCitizen stored as 0/1 not Yes/No
print(f'Issue 2 — SeniorCitizen: {raw_df["SeniorCitizen"].unique()}')
print(f'          Encoded as 0/1 (not Yes/No like other binary cols)')
print(f'          Already numeric — treat as categorical')
print()

# Issue 3: customerID
print(f'Issue 3 — customerID: {raw_df["customerID"].nunique()} unique values')
print(f'          Has no predictive power — must be dropped before training')
print()

# Issue 4: Target encoding
print(f'Issue 4 — Churn: values = {raw_df["Churn"].unique()}')
print(f'          Must convert Yes→1, No→0 for model training')
print()

# Issue 5: Some "No internet service" could be consolidated
cols_with_no_internet = ['OnlineSecurity','OnlineBackup','DeviceProtection',
                         'TechSupport','StreamingTV','StreamingMovies']
print(f'Issue 5 — "No internet service" label in {len(cols_with_no_internet)} columns')
print(f'          These are effectively "No" — can be consolidated during preprocessing')

---
## 🔄 7. Duplicate Records

In [ ]:
# Check duplicates
total_rows   = len(df)
dup_rows     = df.duplicated().sum()
dup_cust_id  = df['customerID'].duplicated().sum()

print('🔍 Duplicate Records Report:')
print('=' * 45)
print(f'  Total rows              : {total_rows:,}')
print(f'  Fully duplicate rows    : {dup_rows}')
print(f'  Duplicate customerIDs   : {dup_cust_id}')

if dup_rows == 0:
    print('\n  ✅ No duplicate rows found — dataset is clean on this front.')
else:
    print(f'\n  ⚠️  Found {dup_rows} duplicate rows — must drop before training.')

---
## 📈 8. Churn Patterns & Correlations

In [ ]:
# Tenure vs Monthly Charges — scatter coloured by Churn
fig, ax = plt.subplots(figsize=(12, 7))

for churn_val, color in CHURN_COLORS.items():
    subset = df[df['Churn'] == churn_val]
    ax.scatter(subset['tenure'], subset['MonthlyCharges'],
               c=color, label=f'Churn={churn_val}', alpha=0.4, s=15)

ax.set_title('Tenure vs Monthly Charges (coloured by Churn)', fontsize=14, fontweight='bold')
ax.set_xlabel('Tenure (months)', fontsize=12)
ax.set_ylabel('Monthly Charges ($)', fontsize=12)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('../logs/tenure_vs_charges.png', dpi=150, bbox_inches='tight')
plt.show()

print('💡 INSIGHT: Churned customers cluster in the LOW tenure + HIGH monthly charges zone.')
print('           → Customers are leaving early when bills are high.')

In [ ]:
# Correlation heatmap — numerical features
df_encoded = df.copy()
df_encoded['Churn_num'] = (df_encoded['Churn'] == 'Yes').astype(int)
df_encoded['SeniorCitizen'] = df_encoded['SeniorCitizen'].astype(int)

corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn_num']
corr_matrix = df_encoded[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
            mask=mask, ax=ax, linewidths=0.5,
            annot_kws={'size': 12, 'weight': 'bold'})
ax.set_title('Correlation Matrix — Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../logs/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('💡 INSIGHTS from Correlation:')
print('  tenure        ↔ Churn_num : Negative (-) → longer tenure = less churn')
print('  MonthlyCharges↔ Churn_num : Positive (+) → higher charges = more churn')
print('  tenure        ↔ TotalCharges: High positive → expected (longer = more total paid)')
print('  TotalCharges  ↔ MonthlyCharges: High positive → multicollinearity risk!')

In [ ]:
# Churn rate by Contract × Internet Service — heatmap
pivot = df.groupby(['Contract', 'InternetService'])['Churn'].apply(
    lambda x: round((x == 'Yes').sum() / len(x) * 100, 1)
).unstack()

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, annot_kws={'size': 13, 'weight': 'bold'})
ax.set_title('Churn Rate (%) by Contract Type × Internet Service', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../logs/contract_internet_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('💡 INSIGHT: Month-to-month + Fiber optic = highest risk segment (~52% churn!)')

In [ ]:
# Tenure distribution by contract type
fig, ax = plt.subplots(figsize=(12, 5))
contract_types = df['Contract'].unique()
colors_contract = ['#e74c3c', '#3498db', '#2ecc71']

for contract, color in zip(sorted(contract_types), colors_contract):
    subset = df[df['Contract'] == contract]['tenure']
    ax.hist(subset, bins=30, alpha=0.6, color=color, label=contract, density=True)

ax.set_title('Tenure Distribution by Contract Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Density')
ax.legend(title='Contract')
plt.tight_layout()
plt.savefig('../logs/tenure_by_contract.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Monthly charges distribution by churn
fig, ax = plt.subplots(figsize=(12, 5))
for churn_val, color in CHURN_COLORS.items():
    subset = df[df['Churn'] == churn_val]['MonthlyCharges']
    ax.hist(subset, bins=40, alpha=0.6, color=color, label=f'Churn={churn_val}', density=True)
ax.axvline(df[df['Churn']=='Yes']['MonthlyCharges'].mean(), color='#c0392b',
           linestyle='--', linewidth=2, label='Mean (Churned)')
ax.axvline(df[df['Churn']=='No']['MonthlyCharges'].mean(), color='#27ae60',
           linestyle='--', linewidth=2, label='Mean (Not Churned)')
ax.set_title('Monthly Charges Distribution by Churn', fontsize=14, fontweight='bold')
ax.set_xlabel('Monthly Charges ($)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig('../logs/monthly_charges_by_churn.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📋 9. Full Dataset Statistics Summary

In [ ]:
print('📊 Complete Dataset Summary')
print('=' * 60)
print(f'  Total rows             : {len(df):,}')
print(f'  Total columns          : {df.shape[1]}')
print(f'  Numerical columns      : {len(numerical_cols)}')
print(f'  Categorical columns    : {len(categorical_cols)}')
print(f'  Target column          : Churn (Yes/No)')
print(f'  Churn rate             : {(df["Churn"] == "Yes").mean()*100:.1f}%')
print(f'  Missing values         : {df.isnull().sum().sum()} (after TotalCharges fix)')
print(f'  Duplicate rows         : {df.duplicated().sum()}')
print()
print('📋 Categorical column value counts:')
for col in categorical_cols:
    print(f'  {col:<25}: {df[col].nunique()} unique → {list(df[col].unique())}')

---
## 🏆 10. Key EDA Insights Summary

### ✅ Data Quality Findings
| Issue | Detail | Fix |
|-------|--------|-----|
| `TotalCharges` dtype | Stored as string (object) | `pd.to_numeric(errors='coerce')` |
| `TotalCharges` blanks | 11 rows with `" "` (new customers) | Drop those rows |
| `customerID` | No predictive value | Drop column |
| `Churn` encoding | Yes/No strings | Map to 1/0 |
| `SeniorCitizen` | Already 0/1 but needs consistent handling | Treat as categorical |
| Duplicate rows | None found | — |
| Class imbalance | ~26% churn vs ~74% no-churn | Use stratified split + F1 metric |

### 🔑 Top Churn Predictors (Ranked by Insight)
| Rank | Feature | Key Finding |
|------|---------|------------|
| 1 | Contract type | Month-to-month → 43% churn; Two-year → 3% churn |
| 2 | Online Security | No security → 42% churn |
| 3 | Tech Support | No support → 42% churn |
| 4 | Internet Service | Fiber optic → 42% churn |
| 5 | Payment Method | Electronic check → 45% churn |
| 6 | Tenure | Low tenure = high churn risk |
| 7 | Monthly Charges | Higher bill = more churn |
| 8 | Senior Citizen | Seniors churn 42% vs 24% |

### 🎯 Preprocessing Decisions Informed by EDA
1. **Drop** `customerID` — no predictive value
2. **Fix** `TotalCharges` — convert to numeric, drop 11 null rows
3. **Encode** `Churn` as 1/0
4. **One-Hot Encode** all categorical columns
5. **Scale** `tenure`, `MonthlyCharges`, `TotalCharges` with StandardScaler
6. **Stratified train/test split** to maintain churn ratio
7. **Use F1-score** as primary metric (not Accuracy) due to class imbalance
8. **Watch for multicollinearity** between `TotalCharges` and `tenure × MonthlyCharges`